In [1]:
import torch
import transformers

In [2]:
x = torch.tril(torch.ones(3,3))

In [3]:
x.masked_fill(x == 0, float('-inf'))

tensor([[1., -inf, -inf],
        [1., 1., -inf],
        [1., 1., 1.]])

In [4]:
def from_pretrained(model_type):
    # Loading a pretrained model from hugging face.
    assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
    from transformers import GPT2LMHeadModel

    print(f'loading weights from pretrain model {model_type}')

    config_args = {
        "gpt2": dict(n_layer=12, n_head=12, n_embd=768),
        "gpt2-medium": dict(n_layer=24, n_head=16, n_embd=1024),
        "gpt2-large": dict(n_layer=36, n_head=20, n_embd=1280),
        "gpt2-xl": dict(n_layer=48, n_head=25, n_embd=1600),
    }[model_type]
    config_args['vocab_size'] = 50257
    config_args['block_size'] = 1024

    # config = GPTConfig(**config_args)
    # model = GPT(config)
    # sd = model.state_dict()
    # sd_keys = sd.keys()
    # sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # Remove stencil. Not a param.

    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias') and not k.endswith('.attn.bias')]
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

    # assert len(sd_keys_hf) == len(sd_keys), f'states dicts don\'t match.'
    # for k in sd_keys_hf:
    #     if any(k.endswith(w) for w in transposed):
    #         # Conv1D weights.
    #         assert sd_hf[k].shape[::-1] == sd[k].shape
    #         with torch.no_grad():
    #             sd[k].copy_(sd_hf[k].t())
    #     else:
    #         assert sd_hf[k].shape == sd[k].shape
    #         with torch.no_grad():
    #             sd[k].copy_(sd_hf[k])

    return sd_hf

In [5]:
sd_hf = from_pretrained('gpt2')

loading weights from pretrain model gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [6]:
sd_hf.keys()

odict_keys(['transformer.wte.weight', 'transformer.wpe.weight', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.attn.c_attn.weight', 'transformer.h.0.attn.c_attn.bias', 'transformer.h.0.attn.c_proj.weight', 'transformer.h.0.attn.c_proj.bias', 'transformer.h.0.ln_2.weight', 'transformer.h.0.ln_2.bias', 'transformer.h.0.mlp.c_fc.weight', 'transformer.h.0.mlp.c_fc.bias', 'transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias', 'transformer.h.1.ln_1.weight', 'transformer.h.1.ln_1.bias', 'transformer.h.1.attn.c_attn.weight', 'transformer.h.1.attn.c_attn.bias', 'transformer.h.1.attn.c_proj.weight', 'transformer.h.1.attn.c_proj.bias', 'transformer.h.1.ln_2.weight', 'transformer.h.1.ln_2.bias', 'transformer.h.1.mlp.c_fc.weight', 'transformer.h.1.mlp.c_fc.bias', 'transformer.h.1.mlp.c_proj.weight', 'transformer.h.1.mlp.c_proj.bias', 'transformer.h.2.ln_1.weight', 'transformer.h.2.ln_1.bias', 'transformer.h.2.attn.c_attn.weight', 'transformer.h.2.attn.